<h2> Imports & Configuration </h2>

> 🧭 **Modern Data Engineering Ecosystem Navigation**
> - 📘 **Architecture & Concept:** [Data System Design — Storage Optimization & Clustering](https://github.com/santoshjammi29/data-system-design-handbook/tree/main/modules/04_storage_optimization_and_clustering)
> - 💻 **Hands-On Recipe:** [PySpark Handbook — Notebook 08_03: Data Skew Mitigation](https://github.com/santoshjammi29/pyspark-handbook/blob/main/notebooks/08_performance_and_tuning/08_03_data_skew_mitigation_salting.py)
> - 🔬 **Empirical Lab:** [Spark Experiments — 1_data_skew: AQE & Broadcast Join](https://github.com/santoshjammi29/spark-experiments/blob/main/spark/1_data_skew/3_solving_data_skew_aqe_broadcast.ipynb) *(You are here)*
> - 🚨 **Incident Runbook:** [PySpark Runbook 03: Data Skew & Two-Phase Salting](https://github.com/santoshjammi29/pyspark-handbook/blob/main/runbooks/03_data_skew_and_two_phase_salting.md)
> - 🎯 **Interview Defense:** [Interview Handbook — Pillar 01: Spark & Databricks Q12](https://github.com/santoshjammi29/data-engineering-interview-handbook/blob/main/pillars/01_spark_and_databricks.md#q12)
> - 🌐 **Interactive Web Practice:** [Practice on moderndataengineer.uk](https://moderndataengineer.uk/practice/pyspark?topic=data-skew)
> - 📐 **Architecture Blueprint:** [Data Skew Mitigation & Key Salting Architecture](https://moderndataengineer.uk/whiteboards?id=data_skew_salting_repartition)


In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
import time

In [2]:
from pyspark.sql.types import IntegerType
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master("local[4]").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

23/07/09 09:02:54 WARN Utils: Your hostname, Afaques-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.4 instead (on interface en0)
23/07/09 09:02:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/07/09 09:02:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# spark.conf.set("spark.sql.shuffle.partitions", "3")
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Join Skews

In [5]:
transactions_file = "../../data/data_skew/transactions.parquet"
customer_file = "../../data/data_skew/customers.parquet"

df_transactions = spark.read.parquet(transactions_file)
df_customers = spark.read.parquet(customer_file)

In [6]:
df_transactions.printSchema()
df_transactions.show(5, False)

root
 |-- cust_id: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- city: string (nullable = true)



+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
|cust_id   |start_date|end_date  |txn_id         |date      |year|month|day|expense_type |amt   |city       |
+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
|C0YDPQWPBJ|2010-07-01|2018-12-01|TZ5SMKZY9S03OQJ|2018-10-07|2018|10   |7  |Entertainment|10.42 |boston     |
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYIAPPNU066CJ5R|2016-03-27|2016|3    |27 |Motor/Travel |44.34 |portland   |
|C0YDPQWPBJ|2010-07-01|2018-12-01|TETSXIK4BLXHJ6W|2011-04-11|2011|4    |11 |Entertainment|3.18  |chicago    |
|C0YDPQWPBJ|2010-07-01|2018-12-01|TQKL1QFJY3EM8LO|2018-02-22|2018|2    |22 |Groceries    |268.97|los_angeles|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYL6DFP09PPXMVB|2010-10-16|2010|10   |16 |Entertainment|2.66  |chicago    |
+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
only showi

In [7]:
df_customers.printSchema()
df_customers.show(5, False)

root
 |-- cust_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthday: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- city: string (nullable = true)

+----------+-------------+---+------+----------+-----+-----------+
|cust_id   |name         |age|gender|birthday  |zip  |city       |
+----------+-------------+---+------+----------+-----+-----------+
|C007YEYTX9|Aaron Abbott |34 |Female|7/13/1991 |97823|boston     |
|C00B971T1J|Aaron Austin |37 |Female|12/16/2004|30332|chicago    |
|C00WRSJF1Q|Aaron Barnes |29 |Female|3/11/1977 |23451|denver     |
|C01AZWQMF3|Aaron Barrett|31 |Male  |7/9/1998  |46613|los_angeles|
|C01BKUFRHA|Aaron Becker |54 |Male  |11/24/1979|40284|san_diego  |
+----------+-------------+---+------+----------+-----+-----------+
only showing top 5 rows



In [8]:
(
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .show(5, False)
)

+----------+--------+
|cust_id   |ct      |
+----------+--------+
|C0YDPQWPBJ|17539732|
|CBW3FMEAU7|7999    |
|C3KUDEN3KO|7999    |
|C89FCEGPJP|7999    |
|CHNFNR89ZV|7998    |
+----------+--------+
only showing top 5 rows



### 📊 Visual Evidence: Measured Key Skew Distribution
![Measured Key Distribution: Severe Skew Outlier](../../docs/images/generated/skew_key_distribution.png)

> **Key Observation**:
> - Customer `C0YDPQWPBJ` accounts for **17,539,732 rows (44.08%)** of the 39,790,092 total transaction records.
> - The remaining 4,999 customers average only ~4,451 rows each.
> - In standard hash partitioning (`hash(cust_id) % num_partitions`), all 17.5M records land on a single partition, creating an extreme straggler task.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../../docs/images/generated/skew_key_distribution.png"))


In [9]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [10]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [11]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

39790092

time taken: 11.315901279449463


# Using AQE

In [12]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewedJoin.enabled", "true")

In [13]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [14]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

39790092

time taken: 9.6234769821167


# Using Broadcast Joins

In [15]:
# 10MB = 10485760 Bytes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

In [16]:
df_txn_details = (
    df_transactions.join(
        F.broadcast(df_customers),
        on="cust_id",
        how="inner"
    )
)

In [17]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

39790092

time taken: 2.7766311168670654


### 📈 Measured Benchmark Comparison: Skew Mitigation Strategies
![Measured Join Execution Runtime Comparison](../../docs/images/generated/skew_mitigation_benchmark.png)

| Join Strategy | Spark Configuration | Measured Runtime | Speedup vs SMJ | Bottleneck / Trade-off |
| :--- | :--- | :--- | :--- | :--- |
| **SortMergeJoin (Baseline)** | `spark.sql.adaptive.enabled = false` | **11.32s** | Baseline (1.0x) | Single executor core blocked processing 17.5M rows; high disk spill risk |
| **SortMergeJoin + AQE** | `spark.sql.adaptive.enabled = true`<br>`skewJoin.enabled = true` | **9.62s** | **15.0% faster** | Automatically splits skewed partition into smaller sub-partitions |
| **BroadcastHashJoin (BHJ)** | `spark.sql.autoBroadcastJoinThreshold = 10MB` | **2.78s** | **75.5% faster (4.07x)** | Zero shuffle required; dimension table replicated to each executor |

#### 💡 Diagnostic Takeaways:
1. **SortMergeJoin Stragglers**: When shuffle partitions contain severe key skew, the job runtime is bounded by the slowest task processing the hot key.
2. **AQE Skew Join**: Spark 3.x Adaptive Query Execution detects partition sizes post-shuffle and splits oversized partitions into sub-partitions, balancing task workload dynamically without changing user code.
3. **Broadcast Hash Join Dominance**: When the dimension table (`customers`) is small (5,000 rows, ~120 KB), broadcasting it entirely eliminates the shuffle phase, cutting runtime from 11.32s down to 2.78s.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../../docs/images/generated/skew_mitigation_benchmark.png"))


In [18]:
# spark.stop()